# Full Run: Explain-All Pipeline (BGL & HDFS)

Crash-resilient full run with:
- **Incremental JSONL save** — each explanation appended to disk immediately
- **Resume from crash** — counts existing lines and skips completed sessions
- **Sub-range test mode** — cap anomalies to validate pipeline before full run
- **Progress logging** — rate/ETA every 100 sessions

## Workflow
### Sub-range test (recommended first)
1. Set `DATASET`, `MAX_ANOMALIES = 2000` in Cell 2
2. Run cells 1–7 (setup → full run)
3. Run cell 9 (metrics) — verify pass rate ≈ 100%

### Full baseline run
1. Set `MAX_ANOMALIES = None` in Cell 2
2. Run cells 1–7
3. If WSL crashes: restart kernel, run cells 1–6, then cell 8 (resume)
4. Run cell 9 (final metrics)

## Cell 1: Imports

In [1]:
import sys, os, json, time
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Find project root (contains src/ and configs/) — idempotent across re-runs
# Search from CWD upward; fall back to known workspace path
_candidates = [Path(".").resolve()]
_candidates += list(_candidates[0].parents)
_candidates.append(Path.home() / "agentic-log-explanations")  # fallback

project_root = None
for _c in _candidates:
    if (_c / "src").is_dir() and (_c / "configs").is_dir():
        project_root = _c
        break
assert project_root is not None, "Cannot find project root"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

from src.data_loader import BGLDataLoader, HDFSDataLoader
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore, EvidenceDoc, build_evidence_store
from src.retriever import Retriever
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, Signature, ExplanationResult
from src.llm_client import LLMClient
from src.verifier import Verifier
from src.normalizer import get_normalizer

print("All imports OK")

Working directory: /home/dave/agentic-log-explanations
All imports OK


## Cell 2: Configuration

**Change `DATASET`** to switch between BGL and HDFS.
**Change `MAX_ANOMALIES`** to control run scope:
- `2000` — sub-range test (validates pipeline end-to-end)
- `None` — full baseline run

In [2]:
# ===================== CHANGE THIS =====================
DATASET = "BGL"           # "BGL" or "HDFS"
LLM_MODEL = "llama3.1:8b"
MAX_SESSIONS = None       # None = all test sessions  (caps screening input)
MAX_ANOMALIES = 500       # None = all anomalies      (caps explain loop)
MAX_NORMAL_EVIDENCE = 20_000  # Cap normal docs in evidence store (None = all)
#   Sub-range test:  MAX_ANOMALIES = 100,  MAX_NORMAL_EVIDENCE = 20_000
#   Full baseline:   MAX_ANOMALIES = None, MAX_NORMAL_EVIDENCE = 20_000
# =======================================================

# Dataset-specific paths
CONFIGS = {
    "BGL": {
        "log_file": "./logs/BGL.log",
        "label_file": None,
        "model_path": "./best_model/best_model_20250724_072857.pth",
        "patterns_file": "./patterns/bgl_patterns.json",
        "output_dir": "./results",
    },
    "HDFS": {
        "log_file": "./logs/HDFS.log",
        "label_file": "./logs/anomaly_label_HDFS.csv",
        "model_path": "./best_model_HDFS/best_model_HDFS20250804_201746.pth",
        "patterns_file": "./patterns/hdfs_patterns.json",
        "output_dir": "./results_HDFS",
    }
}

cfg = CONFIGS[DATASET]
OUTPUT_DIR = Path(cfg["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RAG settings
TOP_K_ANOMALY = 4
TOP_K_NORMAL = 1

print(f"Dataset:            {DATASET}")
print(f"Output:             {OUTPUT_DIR}")
print(f"Model:              {LLM_MODEL}")
print(f"MAX_ANOMALIES:      {MAX_ANOMALIES or 'all'}")
print(f"MAX_NORMAL_EVIDENCE: {MAX_NORMAL_EVIDENCE or 'all'}")

Dataset:            BGL
Output:             results
Model:              llama3.1:8b
MAX_ANOMALIES:      500
MAX_NORMAL_EVIDENCE: 20000


## Cell 3: Load Data & Screener

In [3]:
# 1. Load data
print(f"[1/2] Loading {DATASET} data...")
if DATASET == "BGL":
    data_loader = BGLDataLoader(log_file=cfg["log_file"])
else:
    data_loader = HDFSDataLoader(
        log_file=cfg["log_file"],
        label_file=cfg["label_file"]
    )
data_loader.load()
data_loader.print_stats()

# 2. Load screener
print(f"\n[2/2] Loading Screener...")
screener = Screener.from_pretrained(
    dataset=DATASET,
    model_path=cfg["model_path"]
)
print("Done.")

[1/2] Loading BGL data...
Loading BGL logs from: logs/BGL.log


Reading BGL logs: 4747963it [00:01, 3455827.62it/s]


Loaded 4747963 log lines


Creating sessions: 100%|██████████| 474796/474796 [00:02<00:00, 192390.84it/s]



BGL Dataset Statistics

TRAIN:
  Total sessions: 332,356
  Normal: 305,041 | Anomaly: 27,315
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

VAL:
  Total sessions: 71,219
  Normal: 65,366 | Anomaly: 5,853
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

TEST:
  Total sessions: 71,221
  Normal: 65,367 | Anomaly: 5,854
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

[2/2] Loading Screener...
Loading Screener for BGL on cuda
Loading cl100k_base (GPT-4) tokenizer...
Loading model weights from: ./best_model/best_model_20250724_072857.pth
Model loaded! Parameters: 13,445,922
Done.


## Cell 4: Screen Test Set → Get Anomalies

In [4]:
# Get test sessions
test_sessions = data_loader.get_test()
if MAX_SESSIONS:
    test_sessions = test_sessions[:MAX_SESSIONS]
print(f"Test sessions: {len(test_sessions):,}")

# Screen all
print("Screening...")
screener_outputs = screener.screen_sessions(test_sessions)

# Collect anomalies (maintain order)
anomaly_sessions = []
anomaly_outputs = []
for session, output in zip(test_sessions, screener_outputs):
    if output.is_anomaly:
        anomaly_sessions.append(session)
        anomaly_outputs.append(output)

total_anomalies = len(anomaly_sessions)
print(f"Predicted anomalies: {total_anomalies:,} / {len(test_sessions):,} "
      f"({total_anomalies/len(test_sessions):.1%})")

# Cap anomalies for sub-range test
if MAX_ANOMALIES and MAX_ANOMALIES < total_anomalies:
    anomaly_sessions = anomaly_sessions[:MAX_ANOMALIES]
    anomaly_outputs = anomaly_outputs[:MAX_ANOMALIES]
    print(f"  → Sub-range mode: capped to {MAX_ANOMALIES:,} anomalies")

# Quick ground-truth check
tp = sum(1 for s in anomaly_sessions if s.label == 1)
fn = sum(1 for s, o in zip(test_sessions, screener_outputs) if s.label == 1 and not o.is_anomaly)
fp = sum(1 for s in anomaly_sessions if s.label == 0)
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}")

Test sessions: 71,221
Screening...


Screening sessions: 100%|██████████| 8903/8903 [00:48<00:00, 181.77it/s]

Predicted anomalies: 6,295 / 71,221 (8.8%)
  → Sub-range mode: capped to 500 anomalies
  TP=477  FP=23  FN=10


## Cell 5: Build Evidence Store + Retriever + Signature Cards

In [5]:
import random

# Evidence store
evidence_path = OUTPUT_DIR / f"evidence_store_{DATASET}.json"
if evidence_path.exists():
    print(f"Loading evidence store from {evidence_path}")
    evidence_store = EvidenceStore(DATASET)
    evidence_store.load(str(evidence_path))
else:
    print(f"Building evidence store...")
    evidence_store = build_evidence_store(
        data_loader, DATASET, save_path=str(evidence_path)
    )

# Sample normal docs if evidence store is very large (BGL has 305K normals)
n_total = len(evidence_store.documents)
if MAX_NORMAL_EVIDENCE:
    anom_docs = [d for d in evidence_store.documents if d.metadata.get("label") == 1]
    norm_docs = [d for d in evidence_store.documents if d.metadata.get("label") == 0]
    sig_docs  = [d for d in evidence_store.documents
                 if d.metadata.get("label") not in (0, 1)]  # signatures etc.

    if len(norm_docs) > MAX_NORMAL_EVIDENCE:
        random.seed(42)
        norm_docs = random.sample(norm_docs, MAX_NORMAL_EVIDENCE)
        evidence_store.documents = anom_docs + norm_docs + sig_docs
        evidence_store._id_to_doc = {d.evidence_id: d for d in evidence_store.documents}
        print(f"Sampled evidence store: {n_total:,} → {len(evidence_store.documents):,} "
              f"(kept {len(anom_docs):,} anomaly + {len(norm_docs):,} normal)")
    else:
        print(f"Evidence store: {n_total:,} documents (no sampling needed)")
else:
    print(f"Evidence store: {n_total:,} documents")

# Load signature cards from patterns JSON
patterns_file = Path(cfg["patterns_file"])
if patterns_file.exists():
    with open(patterns_file) as f:
        patterns = json.load(f)
    for pid, info in patterns.items():
        pattern_key = info.get('merge_key', info.get('fingerprint', 'N/A'))
        sig_text = (f"ERROR SIGNATURE: {info['name']}\n"
                    f"Description: {info['description']}\n\n"
                    f"Key Indicators: {', '.join(info['keywords'])}\n"
                    f"Frequency: {info['frequency']} occurrences\n"
                    f"Fingerprint: {pattern_key}")
        doc = EvidenceDoc(
            evidence_id=f"E_SIG_{pid}",
            session_id=pid,
            text=sig_text,
            evidence_type="signature",
            metadata={"label": 1, "dataset": DATASET,
                      "signature_name": info['name'],
                      "frequency": info['frequency'],
                      "keywords": info['keywords']}
        )
        evidence_store.documents.append(doc)
        evidence_store._id_to_doc[doc.evidence_id] = doc
    print(f"Added {len(patterns)} signature cards → {len(evidence_store.documents):,} total")
else:
    print(f"No patterns file at {patterns_file}")

# Build retriever
print("Building retriever index...")
retriever = Retriever(evidence_store, method="bm25")
retriever.build_index()
print("Done.")

Loading evidence store from results/evidence_store_BGL.json
Evidence store loaded from results/evidence_store_BGL.json (332356 documents)
Sampled evidence store: 332,356 → 47,315 (kept 27,315 anomaly + 20,000 normal)
Added 34 signature cards → 47,349 total
Building retriever index...
Building BM25 index...
BM25 index built with 47349 documents
Done.


## Cell 6: Init LLM Client & Verifier

In [6]:
llm_client = LLMClient(
    provider="ollama",
    model=LLM_MODEL,
    temperature=0.1,
    max_tokens=1024,
    timeout=120
)
if llm_client.is_available():
    print(f"LLM ({LLM_MODEL}) is available")
else:
    print(f"WARNING: LLM not available! Start ollama first.")

prompt_builder = PromptBuilder(dataset=DATASET)
verifier = Verifier(min_keyword_match_ratio=0.0)

print(f"Ready.  (PromptBuilder dataset={DATASET})")

LLM (llama3.1:8b) is available
Ready.  (PromptBuilder dataset=BGL)


---
## Cell 7: Run Pipeline (Test or Full)

Each explanation is appended to the JSONL file **immediately after completion**.
If WSL crashes, you lose nothing — just resume from cell 8.

Output filename includes `_test{N}` when `MAX_ANOMALIES` is set, so test runs
don't overwrite full-run results.

In [7]:
# ── Output file ──
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
sub_tag = f"_test{MAX_ANOMALIES}" if MAX_ANOMALIES else ""
results_file = OUTPUT_DIR / f"explanations_{DATASET}_{run_timestamp}{sub_tag}.jsonl"

# ── Normalizer (post-process LLM signature names) ──
normalizer = get_normalizer(DATASET)

print(f"Starting {'sub-range test' if MAX_ANOMALIES else 'full'} run: {DATASET}")
print(f"Anomalies to explain: {len(anomaly_sessions):,}")
print(f"Output: {results_file}")
print()

# ── Helper: explain one session ──
def explain_session(session, screener_output):
    """Generate explanation for a single session. Returns (result_dict, ExplanationResult)."""
    # Retrieve evidence (mixed: anomaly exemplars + normal contrast)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, top_k_anomaly=TOP_K_ANOMALY, top_k_normal=TOP_K_NORMAL
    )
    
    # Build prompt
    system_prompt, user_prompt = prompt_builder.build_prompt(
        session, screener_output, evidence_hits
    )
    evidence_id_mapping = prompt_builder.build_evidence_id_mapping(session, evidence_hits)
    
    # Call LLM
    parsed_json, llm_response = llm_client.generate_json(
        prompt=user_prompt, system_prompt=system_prompt
    )
    explanation = TraceExplanation.from_dict(parsed_json)
    explanation.raw_response = llm_response.content
    
    # Normalize signature name (strip severity, canonical error types)
    if explanation.signature and explanation.signature.name:
        explanation.signature.name = normalizer.normalize_signature(
            explanation.signature.name
        )
    
    # Build ExplanationResult
    result = ExplanationResult(
        session_id=session.session_id,
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits,
        explanation=explanation,
        evidence_id_mapping=evidence_id_mapping,
        prompt_tokens=llm_response.prompt_tokens,
        completion_tokens=llm_response.completion_tokens,
        total_tokens=llm_response.total_tokens,
        latency_ms=llm_response.latency_ms
    )
    
    # Verify
    query_text = "\n".join(session.lines)
    v = verifier.verify(
        explanation=explanation,
        evidence_hits=evidence_hits,
        evidence_id_mapping=evidence_id_mapping,
        query_session_text=query_text
    )
    
    # Compact dict for JSONL (one line per session)
    record = result.to_dict()
    record["verification_passed"] = v.passed
    record["verification_checks"] = v.total_checks
    record["verification_failed_checks"] = v.failed_checks
    if not v.passed:
        record["verification_issues"] = [i.to_dict() for i in v.issues if i.status.value == "fail"]
    
    return record, result, v


# ── Main loop with incremental save ──
start_time = time.time()
successful = 0
failed = 0
v_passed = 0
v_failed = 0
total_tokens = 0
latencies = []

for idx in tqdm(range(len(anomaly_sessions)), desc="Explaining"):
    session = anomaly_sessions[idx]
    scr_out = anomaly_outputs[idx]
    
    try:
        record, result, v = explain_session(session, scr_out)
        
        # Append to JSONL immediately
        with open(results_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        successful += 1
        total_tokens += result.total_tokens
        latencies.append(result.latency_ms)
        if v.passed:
            v_passed += 1
        else:
            v_failed += 1
    except Exception as e:
        failed += 1
        # Write a failure record so we don't lose the index
        fail_record = {
            "session_id": session.session_id,
            "label": session.label,
            "error": str(e),
            "verification_passed": False
        }
        with open(results_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(fail_record, ensure_ascii=False) + "\n")
        if failed <= 5:  # Only print first 5 errors
            print(f"\n  ✗ {session.session_id}: {e}")
    
    # Progress every 100
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(anomaly_sessions) - idx - 1) / rate
        print(f"\n  [{idx+1}/{len(anomaly_sessions)}] "
              f"rate={rate:.2f}/s  ETA={remaining/60:.0f}min  "
              f"pass={v_passed}  fail={v_failed}  err={failed}")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"DONE: {successful + failed} / {len(anomaly_sessions)} sessions")
print(f"  Successful: {successful}  Failed: {failed}")
print(f"  Verification: {v_passed} passed, {v_failed} failed "
      f"({v_passed/(v_passed+v_failed)*100:.1f}% pass rate)" if (v_passed+v_failed) > 0 else "")
print(f"  Tokens: {total_tokens:,}  Avg: {total_tokens/max(successful,1):.0f}/session")
print(f"  Latency: avg={np.mean(latencies):.0f}ms  p95={np.percentile(latencies,95):.0f}ms" if latencies else "")
print(f"  Wall time: {elapsed:.0f}s ({elapsed/60:.1f}min)")
print(f"  Saved to: {results_file}")

Starting sub-range test run: BGL
Anomalies to explain: 500
Output: results/explanations_BGL_20260212_164743_test500.jsonl



Explaining:  20%|██        | 100/500 [17:58<1:10:08, 10.52s/it]


  [100/500] rate=0.09/s  ETA=72min  pass=100  fail=0  err=0


Explaining:  40%|████      | 200/500 [35:54<53:15, 10.65s/it]  


  [200/500] rate=0.09/s  ETA=54min  pass=200  fail=0  err=0


...


  [300/500] rate=0.09/s  ETA=36min  pass=300  fail=0  err=0


Explaining:  80%|████████  | 400/500 [1:11:36<19:00, 11.41s/it]


  [400/500] rate=0.09/s  ETA=18min  pass=400  fail=0  err=0


Explaining: 100%|██████████| 500/500 [1:29:28<00:00, 10.74s/it]


  [500/500] rate=0.09/s  ETA=0min  pass=500  fail=0  err=0

DONE: 500 / 500 sessions
  Successful: 500  Failed: 0
  Verification: 500 passed, 0 failed (100.0% pass rate)
  Tokens: 1,739,003  Avg: 3478/session
  Latency: avg=9593ms  p95=11611ms
  Wall time: 5368s (89.5min)
  Saved to: results/explanations_BGL_20260212_164743_test500.jsonl


---
## Cell 8: Resume from Crash

After WSL crash:
1. Restart kernel
2. Run cells 1–6 (setup — these are idempotent)
3. Run **this cell** to pick up where we left off

It counts existing lines in the JSONL and resumes from there.
Use this only for **full runs** (`MAX_ANOMALIES = None`).

In [ ]:
# ── Find the most recent partial results file ──
existing_files = sorted(OUTPUT_DIR.glob(f"explanations_{DATASET}_*.jsonl"))
if not existing_files:
    raise FileNotFoundError(f"No partial results found in {OUTPUT_DIR}. Run cell 7 first.")

results_file = existing_files[-1]  # most recent

# Count completed lines
with open(results_file, "r", encoding="utf-8") as f:
    completed_lines = sum(1 for _ in f)

start_idx = completed_lines
remaining = len(anomaly_sessions) - start_idx

print(f"Resume file: {results_file}")
print(f"Completed:   {completed_lines:,} / {len(anomaly_sessions):,}")
print(f"Remaining:   {remaining:,}")

if remaining <= 0:
    print("\nAll sessions already completed! Skip to cell 9.")
else:
    print(f"\nResuming from session index {start_idx}...")
    print()
    
    start_time = time.time()
    successful = 0
    failed = 0
    v_passed = 0
    v_failed = 0
    total_tokens = 0
    latencies = []
    
    for idx in tqdm(range(start_idx, len(anomaly_sessions)),
                    desc="Resuming",
                    initial=start_idx,
                    total=len(anomaly_sessions)):
        session = anomaly_sessions[idx]
        scr_out = anomaly_outputs[idx]
        
        try:
            record, result, v = explain_session(session, scr_out)
            
            with open(results_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
            
            successful += 1
            total_tokens += result.total_tokens
            latencies.append(result.latency_ms)
            if v.passed:
                v_passed += 1
            else:
                v_failed += 1
        except Exception as e:
            failed += 1
            fail_record = {
                "session_id": session.session_id,
                "label": session.label,
                "error": str(e),
                "verification_passed": False
            }
            with open(results_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(fail_record, ensure_ascii=False) + "\n")
            if failed <= 5:
                print(f"\n  ✗ {session.session_id}: {e}")
        
        # Progress every 100
        if (idx + 1) % 100 == 0:
            elapsed = time.time() - start_time
            done_this_run = idx + 1 - start_idx
            rate = done_this_run / elapsed if elapsed > 0 else 0
            eta = (len(anomaly_sessions) - idx - 1) / rate if rate > 0 else 0
            print(f"\n  [{idx+1}/{len(anomaly_sessions)}] "
                  f"rate={rate:.2f}/s  ETA={eta/60:.0f}min  "
                  f"pass={v_passed}  fail={v_failed}  err={failed}")
    
    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"RESUME DONE: {successful + failed} new sessions")
    print(f"  Successful: {successful}  Failed: {failed}")
    print(f"  Verification: {v_passed} passed, {v_failed} failed")
    print(f"  Wall time: {elapsed:.0f}s ({elapsed/60:.1f}min)")
    print(f"  File: {results_file}")
    
    # Final line count
    with open(results_file) as f:
        total_lines = sum(1 for _ in f)
    print(f"  Total lines in file: {total_lines:,} / {len(anomaly_sessions):,}")

---
## Cell 9: Final Metrics & Summary

Read the completed JSONL and compute aggregate metrics.

In [8]:
# Find the results file
sub_tag = f"_test{MAX_ANOMALIES}" if MAX_ANOMALIES else ""
existing_files = sorted(OUTPUT_DIR.glob(f"explanations_{DATASET}_*{sub_tag}.jsonl"))
if not existing_files:
    # Fall back to any results file for this dataset
    existing_files = sorted(OUTPUT_DIR.glob(f"explanations_{DATASET}_*.jsonl"))
results_file = existing_files[-1]

# Read all records
records = []
with open(results_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

run_mode = "sub-range test" if MAX_ANOMALIES else "full run"
print(f"Results file: {results_file}")
print(f"Run mode:     {run_mode}")
print(f"Total records: {len(records):,}")
print()

# Aggregate
n_success = sum(1 for r in records if "error" not in r)
n_error = sum(1 for r in records if "error" in r)
n_v_passed = sum(1 for r in records if r.get("verification_passed", False))
n_v_failed = sum(1 for r in records if not r.get("verification_passed", True) and "error" not in r)

tokens_list = [r["metrics"]["total_tokens"] for r in records if "metrics" in r]
latency_list = [r["metrics"]["latency_ms"] for r in records if "metrics" in r]

# Signature distribution
sig_counts = {}
for r in records:
    sig = r.get("explanation", {}).get("signature", {})
    if sig:
        name = sig.get("name", "UNKNOWN")
        sig_counts[name] = sig_counts.get(name, 0) + 1

print(f"{'='*60}")
print(f"  FINAL METRICS: {DATASET}  ({run_mode})")
print(f"{'='*60}")
print(f"\nExplanations:")
print(f"  Successful: {n_success:,}")
print(f"  Errors:     {n_error:,}")
print(f"\nVerification:")
print(f"  Passed: {n_v_passed:,}")
print(f"  Failed: {n_v_failed:,}")
if n_v_passed + n_v_failed > 0:
    print(f"  Rate:   {n_v_passed/(n_v_passed+n_v_failed)*100:.1f}%")
print(f"\nTokens:")
if tokens_list:
    print(f"  Total: {sum(tokens_list):,}")
    print(f"  Avg:   {np.mean(tokens_list):.0f} / session")
print(f"\nLatency:")
if latency_list:
    print(f"  Avg:   {np.mean(latency_list):.0f} ms")
    print(f"  P95:   {np.percentile(latency_list, 95):.0f} ms")
    print(f"  Total: {sum(latency_list)/1000:.0f}s ({sum(latency_list)/60000:.1f}min)")
print(f"\nSignatures ({len(sig_counts)} unique):")
for name, count in sorted(sig_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {name}: {count:,}")
if len(sig_counts) > 15:
    print(f"  ... and {len(sig_counts)-15} more")

# Save metrics JSON
metrics_path = results_file.with_suffix(".metrics.json")
metrics_out = {
    "dataset": DATASET,
    "run_mode": run_mode,
    "max_anomalies": MAX_ANOMALIES,
    "results_file": str(results_file),
    "counts": {
        "total_anomalies": len(anomaly_sessions),
        "total_test_sessions": len(test_sessions),
        "successful": n_success,
        "errors": n_error,
    },
    "verification": {
        "passed": n_v_passed,
        "failed": n_v_failed,
        "pass_rate": n_v_passed / max(n_v_passed + n_v_failed, 1)
    },
    "tokens": {
        "total": sum(tokens_list) if tokens_list else 0,
        "avg": float(np.mean(tokens_list)) if tokens_list else 0
    },
    "latency": {
        "avg_ms": float(np.mean(latency_list)) if latency_list else 0,
        "p95_ms": float(np.percentile(latency_list, 95)) if latency_list else 0,
        "total_ms": sum(latency_list) if latency_list else 0
    },
    "signatures": sig_counts
}
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"\nMetrics saved to: {metrics_path}")

Results file: results/explanations_BGL_20260212_164743_test500.jsonl
Run mode:     sub-range test
Total records: 500

  FINAL METRICS: BGL  (sub-range test)

Explanations:
  Successful: 500
  Errors:     0

Verification:
  Passed: 500
  Failed: 0
  Rate:   100.0%

Tokens:
  Total: 1,739,003
  Avg:   3478 / session

Latency:
  Avg:   9593 ms
  P95:   11611 ms
  Total: 4796s (79.9min)

Signatures (71 unique):
  KERNEL__FATAL__DATA_TLB_ERROR: 142
  KERNEL__FATAL__DATA_STORAGE_INTERRUPT: 58
  APP__FATAL__CIOD_STREAM_ERROR: 47
  KERNEL__FATAL_data_TLB_ERROR: 28
  KERNEL__FATAL__data TLB error interrupt: 27
  KERNEL__FATAL__LUSTRE_MOUNT_FAILED: 19
  KERNEL__FATAL_data_storage_interrupt: 17
  APP__CIOD_STREAM_ERROR: 13
  KERNEL__FATAL_LUSTRE_MOUNT_FAILED: 12
  KERNEL__FATAL__data_storage_interrupt: 10
  KERNEL__FATAL__BAD_MESSAGE_HEADER: 9
  APP__FATAL__LOGIN_CHDIR_FAILED: 9
  KERNEL__FATAL__data_TLB_error_interrupt: 8
  KERNEL__FATAL__kernel terminated for reason 1001: 7
  KERNEL__FATAL__ker